# Kaggle Tabular Baseline Template — 5 Hour Selection / No Internet

Template untuk tabular classification/regression.

Fitur:
- auto-discover `train.csv`, `test.csv`, `sample_submission.csv`
- auto-detect target/id/task type
- numeric/categorical/date-like preprocessing
- LightGBM/XGBoost/CatBoost jika tersedia, fallback ke sklearn
- KFold/StratifiedKFold
- OOF metric report
- submission generator

Catatan umum:
- Template ini sengaja generic karena detail kompetisi belum diketahui.
- Auto-detect bisa salah. Bagian paling penting adalah cell `CONFIG`.
- Selalu cek `sample_submission.csv`, metric, dan format kolom sebelum final submit.
- Target pertama saat seleksi: buat `submission.csv` valid secepat mungkin.

In [ ]:
import os, re, gc, glob, math, warnings, random
from pathlib import Path
from pprint import pprint
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, log_loss,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

CONFIG = {
    'TRAIN_FILE': None,
    'TEST_FILE': None,
    'SAMPLE_SUBMISSION_FILE': None,
    'TARGET_COL': None,
    'ID_COL': None,
    # None, 'binary', 'multiclass', 'regression'
    'TASK_TYPE': None,
    # None, 'auc', 'accuracy', 'f1', 'logloss', 'rmse', 'mae'
    'PRIMARY_METRIC': None,
    'N_SPLITS': 5,
    'FAST_MODE': True,
    # 'auto', 'lightgbm', 'xgboost', 'catboost', 'sklearn'
    'MODEL': 'auto',
    'SUBMISSION_FILE': 'submission.csv',
}

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists():
    INPUT_ROOT = Path('.')

csv_files = sorted(str(p) for p in INPUT_ROOT.glob('**/*.csv'))
print(f'Found {len(csv_files)} CSV files')
for i, f in enumerate(csv_files):
    try:
        preview = pd.read_csv(f, nrows=3)
        print(f'{i:02d}. {f} | shape_preview={preview.shape} | columns={list(preview.columns)[:12]}')
    except Exception as e:
        print(f'{i:02d}. {f} | cannot preview: {e}')

def pick_file(files, keywords):
    keywords = [k.lower() for k in keywords]
    for f in files:
        name = Path(f).name.lower()
        if any(k in name for k in keywords):
            return f
    return None

if CONFIG['TRAIN_FILE'] is None:
    CONFIG['TRAIN_FILE'] = pick_file(csv_files, ['train'])
if CONFIG['TEST_FILE'] is None:
    CONFIG['TEST_FILE'] = pick_file(csv_files, ['test'])
if CONFIG['SAMPLE_SUBMISSION_FILE'] is None:
    CONFIG['SAMPLE_SUBMISSION_FILE'] = pick_file(csv_files, ['sample', 'submission'])

print('\nSelected:')
pprint({k: CONFIG[k] for k in ['TRAIN_FILE', 'TEST_FILE', 'SAMPLE_SUBMISSION_FILE']})
assert CONFIG['TRAIN_FILE'] is not None, 'Set CONFIG[TRAIN_FILE] manual'
assert CONFIG['TEST_FILE'] is not None, 'Set CONFIG[TEST_FILE] manual'

In [ ]:
train = pd.read_csv(CONFIG['TRAIN_FILE'])
test = pd.read_csv(CONFIG['TEST_FILE'])
sample_submission = pd.read_csv(CONFIG['SAMPLE_SUBMISSION_FILE']) if CONFIG['SAMPLE_SUBMISSION_FILE'] else None

print('train:', train.shape)
print('test :', test.shape)
if sample_submission is not None:
    print('sample_submission:', sample_submission.shape)
    display(sample_submission.head())

display(train.head())
display(test.head())

In [ ]:
def auto_detect_target(train_df, test_df):
    diff = [c for c in train_df.columns if c not in test_df.columns]
    if len(diff) == 1:
        return diff[0]
    for c in ['target', 'label', 'class', 'y', 'score', 'price', 'SalePrice']:
        if c in train_df.columns and c not in test_df.columns:
            return c
    return train_df.columns[-1]

def auto_detect_id(train_df, test_df, sample_df=None):
    common = [c for c in train_df.columns if c in test_df.columns]
    if sample_df is not None:
        first = sample_df.columns[0]
        if first in test_df.columns:
            return first
    for c in ['id', 'ID', 'Id', 'row_id', 'index', 'image_id']:
        if c in common:
            return c
    if common and train_df[common[0]].is_unique and test_df[common[0]].is_unique:
        return common[0]
    return None

if CONFIG['TARGET_COL'] is None:
    CONFIG['TARGET_COL'] = auto_detect_target(train, test)
if CONFIG['ID_COL'] is None:
    CONFIG['ID_COL'] = auto_detect_id(train, test, sample_submission)

target_col = CONFIG['TARGET_COL']
id_col = CONFIG['ID_COL']
assert target_col in train.columns

print('TARGET_COL:', target_col)
print('ID_COL    :', id_col)

y_raw = train[target_col]

def infer_task_type(y):
    if CONFIG['TASK_TYPE'] is not None:
        return CONFIG['TASK_TYPE']
    if y.dtype == 'object' or str(y.dtype).startswith('category') or str(y.dtype) == 'bool':
        return 'binary' if y.nunique() <= 2 else 'multiclass'
    if y.nunique() <= 20 and y.nunique() / len(y) < 0.05:
        return 'binary' if y.nunique() <= 2 else 'multiclass'
    return 'regression'

task_type = infer_task_type(y_raw)
print('TASK_TYPE:', task_type)
print('Target nunique:', y_raw.nunique(dropna=True))
display(y_raw.value_counts(dropna=False).head(20))

In [ ]:
print('Missing values in train:')
display(train.isna().mean().sort_values(ascending=False).head(20))
print('Missing values in test:')
display(test.isna().mean().sort_values(ascending=False).head(20))

if id_col is not None:
    print('ID unique train/test:', train[id_col].is_unique, test[id_col].is_unique)

In [ ]:
drop_cols = [target_col]
if id_col is not None:
    drop_cols.append(id_col)
features = [c for c in train.columns if c not in drop_cols and c in test.columns]
print('Num features:', len(features))

X_train_raw = train[features].copy()
X_test_raw = test[features].copy()

def add_date_features(tr, te):
    tr = tr.copy(); te = te.copy()
    candidates = [c for c in tr.columns if any(k in c.lower() for k in ['date', 'time', 'timestamp'])]
    for c in candidates:
        tr_dt = pd.to_datetime(tr[c], errors='coerce')
        te_dt = pd.to_datetime(te[c], errors='coerce')
        if max(tr_dt.notna().mean(), te_dt.notna().mean()) >= 0.5:
            for df, dt in [(tr, tr_dt), (te, te_dt)]:
                df[c + '_year'] = dt.dt.year
                df[c + '_month'] = dt.dt.month
                df[c + '_day'] = dt.dt.day
                df[c + '_dow'] = dt.dt.dayofweek
                df[c + '_hour'] = dt.dt.hour
            tr.drop(columns=[c], inplace=True)
            te.drop(columns=[c], inplace=True)
    return tr, te

X_train_raw, X_test_raw = add_date_features(X_train_raw, X_test_raw)
combined = pd.concat([X_train_raw, X_test_raw], axis=0, ignore_index=True)
combined['__missing_count__'] = combined.isna().sum(axis=1)

cat_cols = combined.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = [c for c in combined.columns if c not in cat_cols]
print('Numeric:', len(num_cols), 'Categorical:', len(cat_cols))

for c in num_cols:
    combined[c] = pd.to_numeric(combined[c], errors='coerce')
    med = combined[c].median()
    if pd.isna(med): med = 0
    combined[c] = combined[c].fillna(med)

for c in cat_cols:
    combined[c] = combined[c].astype(str).fillna('__MISSING__')
    freq = combined[c].value_counts(normalize=True)
    combined[c + '__freq'] = combined[c].map(freq).astype('float32')
    le = LabelEncoder()
    combined[c] = le.fit_transform(combined[c]).astype('int32')

X = combined.iloc[:len(train)].reset_index(drop=True)
X_test = combined.iloc[len(train):].reset_index(drop=True)
print('Processed:', X.shape, X_test.shape)
display(X.head())

In [ ]:
if task_type in ['binary', 'multiclass']:
    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y_raw.astype(str))
    n_classes = len(target_encoder.classes_)
    print('Classes:', list(target_encoder.classes_))
else:
    target_encoder = None
    y = pd.to_numeric(y_raw, errors='coerce').values
    n_classes = None

In [ ]:
available = {}
try:
    import lightgbm as lgb
    available['lightgbm'] = True
except Exception as e:
    available['lightgbm'] = False
    print('LightGBM unavailable:', repr(e))
try:
    import xgboost as xgb
    available['xgboost'] = True
except Exception as e:
    available['xgboost'] = False
    print('XGBoost unavailable:', repr(e))
try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    available['catboost'] = True
except Exception as e:
    available['catboost'] = False
    print('CatBoost unavailable:', repr(e))

print('Available:', available)

def choose_model_name():
    if CONFIG['MODEL'] != 'auto':
        return CONFIG['MODEL']
    for m in ['lightgbm', 'xgboost', 'catboost']:
        if available.get(m, False):
            return m
    return 'sklearn'

MODEL_NAME = choose_model_name()
print('MODEL_NAME:', MODEL_NAME)

def build_model():
    fast = CONFIG['FAST_MODE']
    if MODEL_NAME == 'lightgbm' and available['lightgbm']:
        if task_type == 'regression':
            return lgb.LGBMRegressor(n_estimators=500 if fast else 1200, learning_rate=0.03, num_leaves=31, subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
        return lgb.LGBMClassifier(n_estimators=500 if fast else 1200, learning_rate=0.03, num_leaves=31, subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    if MODEL_NAME == 'xgboost' and available['xgboost']:
        if task_type == 'regression':
            return xgb.XGBRegressor(n_estimators=400 if fast else 900, learning_rate=0.03, max_depth=6, subsample=0.85, colsample_bytree=0.85, objective='reg:squarederror', tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1)
        return xgb.XGBClassifier(n_estimators=400 if fast else 900, learning_rate=0.03, max_depth=6, subsample=0.85, colsample_bytree=0.85, objective='binary:logistic' if task_type == 'binary' else 'multi:softprob', eval_metric='logloss' if task_type == 'binary' else 'mlogloss', tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1)
    if MODEL_NAME == 'catboost' and available['catboost']:
        if task_type == 'regression':
            return CatBoostRegressor(iterations=400 if fast else 900, learning_rate=0.03, depth=6, loss_function='RMSE', random_seed=RANDOM_STATE, verbose=False)
        return CatBoostClassifier(iterations=400 if fast else 900, learning_rate=0.03, depth=6, loss_function='Logloss' if task_type == 'binary' else 'MultiClass', random_seed=RANDOM_STATE, verbose=False)
    if task_type == 'regression':
        return ExtraTreesRegressor(n_estimators=300 if fast else 600, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
    return ExtraTreesClassifier(n_estimators=300 if fast else 600, min_samples_leaf=2, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)

In [ ]:
def classification_metrics(y_true, proba):
    pred = np.argmax(proba, axis=1)
    out = {'accuracy': accuracy_score(y_true, pred), 'f1_macro': f1_score(y_true, pred, average='macro')}
    try: out['logloss'] = log_loss(y_true, proba)
    except Exception: pass
    try:
        out['auc'] = roc_auc_score(y_true, proba[:, 1]) if proba.shape[1] == 2 else roc_auc_score(y_true, proba, multi_class='ovr')
    except Exception: pass
    return out

def regression_metrics(y_true, pred):
    return {'rmse': mean_squared_error(y_true, pred, squared=False), 'mae': mean_absolute_error(y_true, pred), 'r2': r2_score(y_true, pred)}

def show_metrics(d, prefix=''):
    print(prefix + ' | '.join(f'{k}: {v:.5f}' for k, v in d.items()))

In [ ]:
if task_type in ['binary', 'multiclass']:
    splitter = StratifiedKFold(n_splits=CONFIG['N_SPLITS'], shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(X), n_classes), dtype=np.float32)
    test_pred = np.zeros((len(X_test), n_classes), dtype=np.float32)
else:
    splitter = KFold(n_splits=CONFIG['N_SPLITS'], shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X), dtype=np.float32)
    test_pred = np.zeros(len(X_test), dtype=np.float32)

models = []
for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y if task_type != 'regression' else None), 1):
    print('=' * 80)
    print('Fold', fold)
    model = build_model()
    model.fit(X.iloc[tr_idx], y[tr_idx])
    if task_type in ['binary', 'multiclass']:
        va_proba = model.predict_proba(X.iloc[va_idx])
        te_proba = model.predict_proba(X_test)
        if va_proba.ndim == 1:
            va_proba = np.vstack([1 - va_proba, va_proba]).T
            te_proba = np.vstack([1 - te_proba, te_proba]).T
        oof[va_idx] = va_proba
        test_pred += te_proba / CONFIG['N_SPLITS']
        show_metrics(classification_metrics(y[va_idx], va_proba), f'Fold {fold}: ')
    else:
        va_pred = model.predict(X.iloc[va_idx])
        te_pred = model.predict(X_test)
        oof[va_idx] = va_pred
        test_pred += te_pred / CONFIG['N_SPLITS']
        show_metrics(regression_metrics(y[va_idx], va_pred), f'Fold {fold}: ')
    models.append(model)
    gc.collect()

print('=' * 80)
print('OOF')
show_metrics(classification_metrics(y, oof) if task_type != 'regression' else regression_metrics(y, oof), 'OOF: ')

In [ ]:
best_threshold = 0.5
if task_type == 'binary':
    probs = oof[:, 1]
    thrs = np.linspace(0.05, 0.95, 181)
    best_f1, best_t_f1 = -1, 0.5
    best_acc, best_t_acc = -1, 0.5
    for t in thrs:
        pred = (probs >= t).astype(int)
        f1 = f1_score(y, pred)
        acc = accuracy_score(y, pred)
        if f1 > best_f1:
            best_f1, best_t_f1 = f1, t
        if acc > best_acc:
            best_acc, best_t_acc = acc, t
    print(f'Best F1 threshold={best_t_f1:.3f} f1={best_f1:.5f}')
    print(f'Best Acc threshold={best_t_acc:.3f} acc={best_acc:.5f}')
    if CONFIG['PRIMARY_METRIC'] == 'f1': best_threshold = best_t_f1
    if CONFIG['PRIMARY_METRIC'] == 'accuracy': best_threshold = best_t_acc
print('Selected threshold:', best_threshold)

In [ ]:
def make_submission():
    if sample_submission is not None:
        sub = sample_submission.copy()
        id_candidate = sub.columns[0]
        pred_cols = [c for c in sub.columns if c != id_candidate]
    else:
        sub = pd.DataFrame()
        sub[id_col if id_col is not None else 'id'] = test[id_col].values if id_col is not None else np.arange(len(test))
        pred_cols = ['target']
        sub['target'] = 0

    if task_type == 'regression':
        sub[pred_cols[0]] = test_pred
    elif task_type == 'binary':
        if len(pred_cols) == 1:
            # default probability positive class. For label submission, uncomment below.
            sub[pred_cols[0]] = test_pred[:, 1]
            # lbl = (test_pred[:, 1] >= best_threshold).astype(int)
            # sub[pred_cols[0]] = target_encoder.inverse_transform(lbl)
        else:
            for i, c in enumerate(pred_cols[:test_pred.shape[1]]):
                sub[c] = test_pred[:, i]
    else:
        if len(pred_cols) == test_pred.shape[1]:
            for i, c in enumerate(pred_cols):
                sub[c] = test_pred[:, i]
        else:
            lbl = np.argmax(test_pred, axis=1)
            sub[pred_cols[0]] = target_encoder.inverse_transform(lbl)
    return sub

submission = make_submission()
display(submission.head())
print('shape:', submission.shape, 'NaN:', submission.isna().sum().sum())
submission.to_csv(CONFIG['SUBMISSION_FILE'], index=False)
print('Saved:', CONFIG['SUBMISSION_FILE'])

## Tabular quick improvement list

Setelah submission pertama:
- Pastikan metric: probability vs label.
- Cek leakage dari `id`, tanggal, user/session/group.
- Coba `CONFIG['MODEL']`: `lightgbm`, `xgboost`, `catboost`, `sklearn`.
- Tambah feature datetime / frequency encoding / group aggregate kalau jelas.
- Jika class imbalance parah, cek F1 threshold tuning.